# Comparación de Modelos — GBT vs AdaBoost

Evaluación comparativa de dos algoritmos de **Boosting** para la detección de fraude bancario.

| Modelo | Librería | Tipo de Boosting |
|---|---|---|
| **Gradient Boosting Trees (GBT)** | PySpark MLlib | Gradient Boosting (corrige residuos por gradiente) |
| **AdaBoost** | scikit-learn | Adaptive Boosting (pondera los errores) |

---

## Fundamentos comparativos

### Gradient Boosting Trees (GBT)
Minimiza una función de pérdida por descenso de gradiente. Cada árbol nuevo aprende sobre los **residuos del gradiente**:
$$F_m(x) = F_{m-1}(x) + \nu \cdot h_m(x)$$

### AdaBoost (Adaptive Boosting)
En lugar de trabajar con gradientes, AdaBoost **aumenta el peso** de las muestras mal clasificadas para que el siguiente clasificador se enfoque en ellas:

$$H(x) = \text{sign}\left(\sum_{m=1}^{M} \alpha_m h_m(x)\right)$$

Donde $\alpha_m = \frac{1}{2} \ln\left(\frac{1 - \epsilon_m}{\epsilon_m}\right)$ es el peso del clasificador $m$ según su tasa de error $\epsilon_m$.

### Diferencia clave
| | GBT | AdaBoost |
|---|---|---|
| Estrategia | Minimizar pérdida por gradiente | Ponderar muestras mal clasificadas |
| Sensibilidad a ruido | Moderada | Alta (muy sensible a outliers) |
| Velocidad | Lento (secuencial) | Más rápido |
| Overfitting | Controlable con `stepSize` y `maxDepth` | Puede ocurrir con muchos estimadores |

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score, confusion_matrix)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time

spark = SparkSession.builder.appName("Model_Comparison_Boosting").getOrCreate()

df = spark.read.csv("../data/processed/bank_fraud_clean.csv", header=True, inferSchema=True)
df = df.withColumn("is_fraud", col("is_fraud").cast("double"))

print(f"Dataset cargado: {df.count():,} filas")

Dataset cargado: 221,092 filas


## 1. Preparación de features (compartido)

In [2]:
CATEGORICAL_COLS = ["merchant_category", "payment_method", "device_type"]
NUMERIC_COLS = [
    "hour_of_day", "is_weekend", "is_night_transaction",
    "customer_age", "credit_score", "account_age_years",
    "account_balance", "transaction_amount", "num_prev_transactions",
    "transaction_freq_monthly", "distance_from_home_km",
    "time_since_last_txn_hrs", "is_international",
    "failed_attempts", "pin_changed_recently"
]
TARGET_COL = "is_fraud"

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in CATEGORICAL_COLS
]
feature_cols = NUMERIC_COLS + [c + "_idx" for c in CATEGORICAL_COLS]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  |  Test: {test_df.count():,}")

Train: 177,080  |  Test: 44,012


## 2. Modelo 1 — Gradient Boosting Trees (PySpark)

In [ ]:
gbt = GBTClassifier(
    labelCol=TARGET_COL, featuresCol="features",
    maxIter=100, maxDepth=5, stepSize=0.1,
    subsamplingRate=0.8, seed=42
)
pipeline_gbt = Pipeline(stages=indexers + [assembler, gbt])

print("Entrenando Gradient Boosting Trees (PySpark)...")
t0 = time.time()
model_gbt = pipeline_gbt.fit(train_df)
time_gbt = time.time() - t0
print(f"  ✓ Completado en {time_gbt:.1f}s")

Entrenando Gradient Boosting Trees (PySpark)...


## 3. Modelo 2 — AdaBoost (scikit-learn)

> AdaBoost no está disponible en PySpark MLlib, por lo que se entrena con scikit-learn sobre una muestra representativa del dataset.

In [ ]:
# Preparar datos para AdaBoost:
# Aplicar los indexers del pipeline GBT para transformar categóricas
# y luego convertir a Pandas

# Usamos el pipeline de GBT para indexar las columnas categóricas
indexer_pipeline = Pipeline(stages=indexers)
indexed_train = indexer_pipeline.fit(train_df).transform(train_df)
indexed_test  = indexer_pipeline.fit(train_df).transform(test_df)

ada_feature_cols = NUMERIC_COLS + [c + "_idx" for c in CATEGORICAL_COLS]

# Convertir a Pandas (toda la muestra, el dataset ya fue reducido a ~220k en preprocesamiento)
print("Convirtiendo datos a Pandas para AdaBoost...")
train_pdf = indexed_train.select(ada_feature_cols + [TARGET_COL]).toPandas()
test_pdf  = indexed_test.select(ada_feature_cols + [TARGET_COL]).toPandas()

X_train = train_pdf[ada_feature_cols].values
y_train = train_pdf[TARGET_COL].values.astype(int)
X_test  = test_pdf[ada_feature_cols].values
y_test  = test_pdf[TARGET_COL].values.astype(int)

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# Entrenar AdaBoost
# Usa árboles de decisión poco profundos como clasificadores base (stumps)
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

print("Entrenando AdaBoost (scikit-learn)...")
t0 = time.time()
ada.fit(X_train, y_train)
time_ada = time.time() - t0
print(f"  ✓ Completado en {time_ada:.1f}s")

## 4. Métricas comparativas

In [ ]:
# ── Métricas GBT (PySpark) ─────────────────────────────
pred_gbt = model_gbt.transform(test_df)

acc_gbt = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL, predictionCol="prediction", metricName="accuracy"
).evaluate(pred_gbt)
f1_gbt = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL, predictionCol="prediction", metricName="f1"
).evaluate(pred_gbt)
auc_gbt = BinaryClassificationEvaluator(
    labelCol=TARGET_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(pred_gbt)
prec_gbt = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL, predictionCol="prediction", metricName="weightedPrecision"
).evaluate(pred_gbt)
rec_gbt = MulticlassClassificationEvaluator(
    labelCol=TARGET_COL, predictionCol="prediction", metricName="weightedRecall"
).evaluate(pred_gbt)

# ── Métricas AdaBoost (sklearn) ───────────────────────
y_pred_ada  = ada.predict(X_test)
y_prob_ada  = ada.predict_proba(X_test)[:, 1]

acc_ada  = accuracy_score(y_test, y_pred_ada)
f1_ada   = f1_score(y_test, y_pred_ada, average="weighted")
auc_ada  = roc_auc_score(y_test, y_prob_ada)
prec_ada = precision_score(y_test, y_pred_ada, average="weighted", zero_division=0)
rec_ada  = recall_score(y_test, y_pred_ada, average="weighted")

# ── Tabla comparativa ─────────────────────────────────
results = pd.DataFrame({
    "Métrica":    ["Accuracy", "F1-Score", "AUC-ROC", "Precision", "Recall", "Tiempo (s)"],
    "GBT":        [f"{acc_gbt:.4f}", f"{f1_gbt:.4f}", f"{auc_gbt:.4f}",
                   f"{prec_gbt:.4f}", f"{rec_gbt:.4f}", f"{time_gbt:.1f}"],
    "AdaBoost":   [f"{acc_ada:.4f}", f"{f1_ada:.4f}", f"{auc_ada:.4f}",
                   f"{prec_ada:.4f}", f"{rec_ada:.4f}", f"{time_ada:.1f}"]
})

print("\n" + "=" * 55)
print("         COMPARACIÓN: GBT vs AdaBoost")
print("=" * 55)
print(results.to_string(index=False))
print("=" * 55)

best = "GBT" if auc_gbt >= auc_ada else "AdaBoost"
print(f"\n➡  Mejor modelo por AUC-ROC: {best}")

## 5. Gráfico comparativo de métricas

In [ ]:
metrics_names = ["Accuracy", "F1-Score", "AUC-ROC", "Precision", "Recall"]
vals_gbt = [acc_gbt, f1_gbt, auc_gbt, prec_gbt, rec_gbt]
vals_ada = [acc_ada, f1_ada, auc_ada, prec_ada, rec_ada]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, vals_gbt, width, label="Gradient Boosting Trees",
               color="darkorange", alpha=0.87, edgecolor="white")
bars2 = ax.bar(x + width/2, vals_ada, width, label="AdaBoost",
               color="steelblue", alpha=0.87, edgecolor="white")

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f"{bar.get_height():.3f}", ha="center", va="bottom",
            fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylim([0.5, 1.1])
ax.set_ylabel("Valor", fontsize=12)
ax.set_title("Comparación de Métricas: GBT vs AdaBoost", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Matrices de confusión comparativas

In [ ]:
from sklearn.metrics import confusion_matrix
import os

# SOLUCIÓN WINDOWS: MulticlassMetrics (RDD) falla sin winutils
# Matriz GBT (usamos pandas y sklearn)
pred_pdf_gbt = pred_gbt.select("prediction", col(TARGET_COL).cast("double")).toPandas()
cm_gbt = confusion_matrix(pred_pdf_gbt[TARGET_COL].values, pred_pdf_gbt["prediction"].values).astype(float)

# Matriz AdaBoost (ya es sklearn)
cm_ada = confusion_matrix(y_test, y_pred_ada).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, cm, title, cmap in zip(
    axes,
    [cm_gbt, cm_ada],
    ["Gradient Boosting Trees", "AdaBoost"],
    [plt.cm.Oranges, plt.cm.Blues]
):
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.colorbar(im, ax=ax)
    classes = ["No Fraude", "Fraude"]
    ticks = np.arange(len(classes))
    ax.set_xticks(ticks); ax.set_xticklabels(classes, fontsize=11)
    ax.set_yticks(ticks); ax.set_yticklabels(classes, fontsize=11)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(int(cm[i, j]), ','),
                    ha="center", va="center", fontsize=13, fontweight="bold",
                    color="white" if cm[i, j] > thresh else "black")
    ax.set_ylabel("Etiqueta real", fontsize=11)
    ax.set_xlabel("Predicción", fontsize=11)
    ax.set_title(f"Matriz de Confusión\n{title}", fontsize=12, fontweight="bold")

plt.suptitle("Comparación de Matrices de Confusión", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
os.makedirs("../docs", exist_ok=True)
plt.savefig("../docs/confusion_matrices_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Curvas ROC comparativas

In [ ]:
import os
from sklearn.metrics import roc_curve

# SOLUCIÓN WINDOWS: Deshabilitar Arrow y no usar UDF para extraer probabilidad
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Puntos ROC para GBT (PySpark -> Pandas puro)
pdf_gbt = pred_gbt.select("probability", TARGET_COL).toPandas()
pdf_gbt["prob"] = pdf_gbt["probability"].apply(lambda v: float(v[1]))
fpr_gbt, tpr_gbt, _ = roc_curve(pdf_gbt[TARGET_COL].values, pdf_gbt["prob"].values)

# Puntos ROC para AdaBoost (sklearn)
fpr_ada, tpr_ada, _ = roc_curve(y_test, y_prob_ada)

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(fpr_gbt, tpr_gbt, color="darkorange", lw=2.5,
        label=f"GBT (AUC = {auc_gbt:.4f})")
ax.plot(fpr_ada, tpr_ada, color="steelblue", lw=2.5,
        label=f"AdaBoost (AUC = {auc_ada:.4f})")
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1.2, label="Clasificador aleatorio")
ax.fill_between(fpr_gbt, tpr_gbt, alpha=0.10, color="darkorange")
ax.fill_between(fpr_ada, tpr_ada, alpha=0.10, color="steelblue")
ax.set_xlabel("Tasa de Falsos Positivos (FPR)", fontsize=12)
ax.set_ylabel("Tasa de Verdaderos Positivos (TPR)", fontsize=12)
ax.set_title("Curvas ROC Comparativas\nGBT vs AdaBoost", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs("../docs", exist_ok=True)
plt.savefig("../docs/roc_curves_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Importancia de Features comparativa

In [ ]:
imp_gbt = model_gbt.stages[-1].featureImportances.toArray()
imp_ada = ada.feature_importances_

fi_df = pd.DataFrame({
    "feature": feature_cols,
    "GBT": imp_gbt,
    "AdaBoost": imp_ada
}).sort_values("GBT", ascending=False).head(12)

x = np.arange(len(fi_df))
width = 0.4

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width/2, fi_df["GBT"],     width, label="GBT",      color="darkorange", alpha=0.87)
ax.bar(x + width/2, fi_df["AdaBoost"], width, label="AdaBoost", color="steelblue",  alpha=0.87)
ax.set_xticks(x)
ax.set_xticklabels(fi_df["feature"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Importancia", fontsize=12)
ax.set_title("Top 12 Features Más Importantes — GBT vs AdaBoost", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../docs/feature_importance_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Análisis e interpretación

In [ ]:
print("=" * 60)
print("          ANÁLISIS COMPARATIVO")
print("=" * 60)
print(f"\n  Gradient Boosting Trees:")
print(f"    AUC-ROC   : {auc_gbt:.4f}")
print(f"    F1-Score  : {f1_gbt:.4f}")
print(f"    Accuracy  : {acc_gbt:.4f}")
print(f"    Tiempo    : {time_gbt:.1f}s")
print(f"\n  AdaBoost:")
print(f"    AUC-ROC   : {auc_ada:.4f}")
print(f"    F1-Score  : {f1_ada:.4f}")
print(f"    Accuracy  : {acc_ada:.4f}")
print(f"    Tiempo    : {time_ada:.1f}s")
print("=" * 60)
winner_auc  = "GBT" if auc_gbt  >= auc_ada  else "AdaBoost"
winner_f1   = "GBT" if f1_gbt   >= f1_ada   else "AdaBoost"
winner_time = "GBT" if time_gbt <= time_ada else "AdaBoost"
print(f"  Mejor AUC-ROC  → {winner_auc}")
print(f"  Mejor F1-Score → {winner_f1}")
print(f"  Más rápido     → {winner_time}")
print("=" * 60)

## 10. Conclusiones

### Diferencias observadas

**Gradient Boosting Trees (GBT):**
- Optimiza directamente la función de pérdida logística mediante gradientes, lo que lo hace más preciso en problemas con clases desbalanceadas.
- `subsamplingRate=0.8` introduce aleatoriedad que actúa como regularización implícita.
- Corre sobre el dataset completo con PySpark (procesamiento distribuido).

**AdaBoost:**
- Estrategia más simple: pondera muestras difíciles en cada ronda.
- Más sensible al ruido y outliers que el GBT.
- Al usar árboles de profundidad 3 como base, cada estimador es débil pero juntos forman un clasificador potente.

### Aprendizajes
1. En detección de fraude, el **AUC-ROC** es la métrica más relevante porque captura la capacidad discriminativa del modelo a todos los umbrales.
2. Los **Falsos Negativos (fraudes no detectados)** son más costosos que los Falsos Positivos → preferir mayor Recall en la clase fraude.
3. El GBT con PySpark escala a millones de registros; AdaBoost con sklearn sería inviable en el dataset completo sin muestreo.

### Posibles mejoras
- **Ajuste de hiperparámetros** con CrossValidation + ParamGridBuilder en PySpark.
- **XGBoost** como tercera alternativa (requiere instalar `xgboost`).
- Ajustar el **umbral de clasificación** (threshold) según el costo del negocio.
- Probar **SMOTE** como técnica de balanceo alternativa al undersampling.